# Pashto FastSpeech2: train on the full dataset (Kaggle + Hugging Face Hub)

Training runs in sessions of at most `HOURS` (6 h). Everything a session needs to continue (the newest
checkpoint, logs, configs/metrics, the features) lives in the Hugging Face repo `HF_REPO`: each session pulls it,
trains with a checkpoint at the end of **every epoch** on disk, pushes the newest one to the Hub every
`PUSH_MINUTES`, and stops by itself before `HOURS` (with a final checkpoint and push).

**One-time setup**
1. Hugging Face: *Settings -> Access Tokens -> Create new token*. Fine-grained, under *Org permissions* for
   `speech-itu`: **write access to contents/settings of all repos** and **create repos** (or a classic *write*
   token of an org member who can create repos).
2. Kaggle, in this notebook: *Add-ons -> Secrets -> Add secret*, label `HF_TOKEN`, value = the token; tick it
   so this notebook can read it.
3. Nothing to upload: the first run creates the private repo `HF_REPO` and uploads the alignments
   (`data/pashto_aligned.zip` from the git repo) to it.
4. Right panel: Accelerator **GPU T4 x2**, Internet **on**.

**Smoke test (~30-45 min of GPU quota):** with `SMOKE_TEST = True`, *Run All* in this interactive session. It
creates the repo, preprocesses (first time only; the features are then pushed to the Hub), trains `SMOKE_MINUTES`, pushes to
`smoke/` in the repo every minute, checks a checkpoint downloads back, deletes `smoke/`, and prints a checklist
plus a **time estimate** for the whole training (steps/hour, steps per session, sessions/weeks for 20k/50k/100k
steps). **Stop the session** afterwards (it uses quota while open).

**Real runs:** `SMOKE_TEST = False`, then *Save Version -> Save & Run All (Commit)*. Repeat for every session,
nothing to attach: each one continues from the Hub.

In [ ]:
SMOKE_TEST    = True     # True: short test (interactive). False: a real training session (commit)
SMOKE_MINUTES = 10       # training time of the smoke test
HOURS         = 6        # maximum session length; training stops ~25 min before, then pushes
PUSH_MINUTES  = 30       # push the newest checkpoint to the Hub this often
STEPS         = 100000   # total training steps
BATCH_SIZE    = 16       # lower to 8 on CUDA out-of-memory (30 s clips)
HF_REPO       = "speech-itu/pashto-fastspeech2"

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
REPO_DIR = "/kaggle/temp/pashto-fastspeech2"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/msds25033/pashto-fastspeech2.git {REPO_DIR}
%cd {REPO_DIR}
!git pull -q && git log -1 --format="%h %s"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
SMOKE_ARG = f"--smoke --smoke-minutes {SMOKE_MINUTES}" if SMOKE_TEST else ""
!python kaggle_train.py --hours {HOURS} --push-minutes {PUSH_MINUTES} --steps {STEPS} --batch-size {BATCH_SIZE} --hf-repo {HF_REPO} {SMOKE_ARG}